# 07 · 时序跟踪：Kalman Filter、时间错位与 gating

自动驾驶模型面对的是连续帧，不是相互独立的图片。这个 notebook 用一个 constant-velocity Kalman filter 演示 measurement dropout、时间戳偏移、异常点和 innovation gating 对跟踪质量的影响。

学习目标：

- 明确 state、transition、observation 和 uncertainty 的含义。
- 比较理想同步与时间错位情况下的轨迹 RMSE。
- 用 Mahalanobis-like innovation gate 拒绝明显异常观测。
- 理解真实 tracking 还需要数据关联、ego-motion compensation 和 multi-object management。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.grid'] = True

dt = 0.1
time = np.arange(0.0, 12.0, dt)
true = np.c_[
    0.6 * time + 0.8 * np.sin(0.45 * time),
    1.2 * np.sin(0.32 * time),
]
velocity = np.gradient(true, dt, axis=0)



In [ ]:
class KalmanCV:
    def __init__(self, dt=0.1, process_noise=0.15, measurement_noise=0.35):
        self.dt = dt
        self.x = np.zeros(4)
        self.P = np.eye(4) * 2.0
        self.F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0],
            [0, 0, 0, 1],
        ], dtype=float)
        self.H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], dtype=float)
        self.Q = np.eye(4) * process_noise
        self.R = np.eye(2) * measurement_noise ** 2

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x.copy()

    def update(self, measurement, gate=9.21):
        innovation = measurement - self.H @ self.x
        covariance = self.H @ self.P @ self.H.T + self.R
        score = innovation @ np.linalg.solve(covariance, innovation)
        accepted = score <= gate
        if accepted:
            gain = self.P @ self.H.T @ np.linalg.inv(covariance)
            self.x = self.x + gain @ innovation
            self.P = (np.eye(4) - gain @ self.H) @ self.P
        return self.x.copy(), float(score), bool(accepted)

def run_tracker(measurement_noise=0.35, dropout=0.15, timestamp_offset=0.0, outlier_rate=0.04, gate=9.21, seed=10):
    local = np.random.default_rng(seed)
    shifted_t = np.clip(time + timestamp_offset, time[0], time[-1])
    measurement_truth = np.c_[
        np.interp(shifted_t, time, true[:, 0]),
        np.interp(shifted_t, time, true[:, 1]),
    ]
    measurements = measurement_truth + local.normal(0, measurement_noise, size=true.shape)
    missing = local.random(len(time)) < dropout
    outliers = local.random(len(time)) < outlier_rate
    measurements[outliers] += local.normal(0, 3.5, size=(outliers.sum(), 2))
    tracker = KalmanCV(dt=dt, measurement_noise=measurement_noise)
    estimates, scores, accepted = [], [], []
    for k in range(len(time)):
        estimate = tracker.predict()
        if not missing[k]:
            estimate, score, was_accepted = tracker.update(measurements[k], gate=gate)
        else:
            score, was_accepted = np.nan, False
        estimates.append(estimate[:2])
        scores.append(score)
        accepted.append(was_accepted)
    return measurements, np.asarray(estimates), missing, outliers, np.asarray(scores), np.asarray(accepted)



In [ ]:
measurements, estimates, missing, outliers, scores, accepted = run_tracker()
rmse = np.sqrt(np.mean((estimates - true) ** 2))
print(f'position RMSE: {rmse:.3f} m')
print(f'accepted measurements: {accepted.sum()} / {len(time)}')



In [ ]:
def show_tracking(dropout=0.15, timestamp_offset=0.0, outlier_rate=0.04, gate=9.21):
    measurements, estimates, missing, outliers, scores, accepted = run_tracker(
        dropout=dropout,
        timestamp_offset=timestamp_offset,
        outlier_rate=outlier_rate,
        gate=gate,
    )
    rmse = np.sqrt(np.mean((estimates - true) ** 2))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(true[:, 0], true[:, 1], label='ground truth', linewidth=2)
    ax[0].scatter(measurements[~missing, 0], measurements[~missing, 1], s=8, alpha=0.3, label='measurements')
    ax[0].scatter(measurements[outliers & ~missing, 0], measurements[outliers & ~missing, 1], c='tab:red', s=20, label='outliers')
    ax[0].plot(estimates[:, 0], estimates[:, 1], label='Kalman estimate')
    ax[0].set_title(f'RMSE={rmse:.3f} m')
    ax[0].set_xlabel('x / m')
    ax[0].set_ylabel('y / m')
    ax[0].legend()
    ax[1].plot(time, scores, label='innovation score')
    ax[1].axhline(gate, color='tab:red', linestyle='--', label='gate')
    ax[1].scatter(time[~accepted & ~missing], np.nan_to_num(scores[~accepted & ~missing]), c='tab:red', s=18, label='rejected')
    ax[1].set_title('measurement gating')
    ax[1].set_xlabel('time / s')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

interact(
    show_tracking,
    dropout=FloatSlider(min=0.0, max=0.6, step=0.05, value=0.15, description='dropout'),
    timestamp_offset=FloatSlider(min=-0.25, max=0.25, step=0.025, value=0.0, description='time offset'),
    outlier_rate=FloatSlider(min=0.0, max=0.2, step=0.02, value=0.04, description='outlier'),
    gate=FloatSlider(min=2.0, max=25.0, step=0.5, value=9.21, description='gate'),
);



### 练习：把时间同步当作模型输入的一部分

- 固定 dropout 和 outlier rate，扫描 timestamp_offset，画 RMSE 曲线。
- 调小 gate，观察异常点拒绝率与正常点误拒绝率。
- 加入 ego-motion：让整个坐标系发生旋转或平移，再比较 compensation 前后误差。
- 思考 camera、LiDAR、radar 的时间戳应该在哪一层对齐：采集、数据加载、特征融合还是 tracker。


In [ ]:
offsets = np.linspace(-0.2, 0.2, 9)
rmses = []
for offset in offsets:
    _, estimate, _, _, _, _ = run_tracker(timestamp_offset=float(offset), seed=10)
    rmses.append(np.sqrt(np.mean((estimate - true) ** 2)))
plt.plot(offsets, rmses, marker='o')
plt.axvline(0, color='black', linestyle=':')
plt.xlabel('timestamp offset / s')
plt.ylabel('position RMSE / m')
plt.title('temporal misalignment ablation')
plt.show()
print('best offset:', offsets[int(np.argmin(rmses))])



## 完成标准

至少保留：

- 一张 track / measurement / outlier 图；
- 一张 timestamp offset ablation；
- 一个 gate 的 precision-like 分析：被拒绝的点里有多少是真异常；
- 对真实 tracking 系统还缺什么做出清单：关联、遮挡、轨迹管理、ego motion、传感器时延。
